[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/06-agentic-ai/05-grounding_rag_with_deterministic_matching.ipynb)

In [1]:
# !pip install mbox openai python-dotenv

# Grounding RAG with Deterministic Matching

Retrieval-augmented generation has the same grounding problem `01` described, wearing different clothes. Instead of "does the model remember the right fact," the question becomes "did retrieval pull the right document." When your knowledge base is really a table of named entities, products, accounts, locations, with an article or record attached to each one, the retrieval step is an entity resolution problem first, and a search problem second. This notebook applies the same principle from `01` to that step.

In this notebook you will:

1. Retrieve from a small knowledge base the way a lot of real systems actually do it: literal keyword lookup, before anyone reaches for embeddings
2. Watch that lookup fail outright on a typo, and fail silently on a genuine ambiguity
3. Resolve the entity with M|BOX first, then retrieve, and see both failures disappear
4. Feed the correctly-retrieved article to an LLM and get an answer grounded in the real one

> Note: this notebook makes real calls to the OpenAI API. To run it, put an `OPENAI_API_KEY` in a `.env` file in this directory.

In [2]:
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

## 1. A knowledge base of named entities

`datasets/kb_articles.csv` is five short support articles, one per product. Two of them are a near-duplicate pair on purpose, `Wireless Router X200` and `Wireless Router X200 Pro`, because telling those apart correctly is the actual test of any retrieval approach, not a corner case you can ignore.

In [3]:
kb = pd.read_csv("datasets/kb_articles.csv")
kb

,product_id,product_name,article_text
0,RTR-200,Wireless Router X200,The X200 supports up to 25 connected devices a...
1,RTR-200P,Wireless Router X200 Pro,The X200 Pro supports up to 60 connected devic...
2,CAM-410,Outdoor Security Camera 410,The 410 camera is weatherproofed to an IP66 ra...
3,THM-050,Smart Thermostat T50,The T50 learns your schedule within about a we...
4,LGT-330,Smart Bulb Starter Kit,The starter kit ships with 4 bulbs and one hub...


## 2. Retrieval by literal keyword lookup

Before an embedding index, this is how a lot of tabular knowledge bases actually get searched: check whether the query appears in the product name or article text. It is cheap, it is exact, and it works fine as long as the user types the name correctly.

In [4]:
def naive_lookup(query, kb):
    q = query.lower()
    return kb[kb.apply(lambda r: q in r["product_name"].lower() or q in r["article_text"].lower(), axis=1)]

naive_lookup("wireless router x200", kb)[["product_id", "product_name"]]

,product_id,product_name
0,RTR-200,Wireless Router X200
1,RTR-200P,Wireless Router X200 Pro


That works. Here is the same lookup against what a customer actually typed.

In [5]:
customer_question = "does the wireles rotuer x200 have a longer warranty than a year?"
naive_lookup(customer_question, kb)

,product_id,product_name,article_text


Nothing. Not the right article, not the wrong article, nothing at all. Two typos and the entity the customer is asking about is invisible to this lookup. An agent built on this retrieval step has no article to ground an answer in, and either has to say "I don't know" to a perfectly answerable question, or worse, fall back to the model's own guess about router warranties in general, disconnected from this specific product.

## 3. Resolve first, then retrieve

A raw question is not what you match against a table of entity names anyway, naive or not, a whole sentence looks nothing like `"Wireless Router X200"`. What every retrieval approach actually needs first is the specific product mention inside that question, `"wireles rotuer x200"` here. Getting that span out of a sentence is a separate, solved problem, ask the model to extract it, or use whatever NER step you already have. What happens *after* extraction is what this section is about: index the entity names and resolve that mention to a specific `product_id`. Retrieval then becomes a plain, exact lookup by that id, deterministic once the id is right.

In [6]:
mentioned_product = "wireles rotuer x200"

from mbox.indexing import TableIndexer
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

index = TableIndexer.create_index(kb, index_columns=["product_name"], tmp_dir="tmp_index_kb")

config = TableRecallConfig(
    fields=[TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                    minimum_quality=0, weight=100, mode=TableRecallMode.APPROX)],
    max_results=3, min_total_match_value=0, include_field_scores=True
)

resolved = index.match(queries=pd.DataFrame({"product_name": [mentioned_product]}), config=config)
resolved[["product_name_candidate", "product_name_score"]]

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,product_name_candidate,product_name_score
0,Wireless Router X200,73
1,Wireless Router X200 Pro,70


`Wireless Router X200` resolves correctly, ahead of its own near-duplicate `Pro` sibling, despite the typos. That score, not a substring check, is what retrieval should be conditioned on.

In [7]:
best = resolved.iloc[0]
article = kb.loc[kb["product_name"] == best["product_name_candidate"], "article_text"].iloc[0]
print(f"Resolved to: {best['product_name_candidate']} (score {best['product_name_score']})\n")
print(article)

Resolved to: Wireless Router X200 (score 73)

The X200 supports up to 25 connected devices and ships with a 1-year warranty. Firmware updates install automatically overnight.


## 4. Ground the LLM's answer in the correct article

Now the model answers from the one article that actually describes this product, not from whatever it can guess about routers in general.

In [8]:
from openai import OpenAI
client = OpenAI()

prompt = f"Support article:\n{article}\n\nCustomer question: {customer_question}\n\nAnswer using only the article above."
response = client.chat.completions.create(model="gpt-4o", messages=[{"role": "user", "content": prompt}])
print(response.choices[0].message.content)

No, the X200 ships with a 1-year warranty.


## 5. The failure that isn't a typo: genuine ambiguity

Not every retrieval miss is a spelling problem. Ask about "the x200" without saying which one, and watch the naive lookup this time.

In [9]:
naive_lookup("x200", kb)[["product_id", "product_name"]]

,product_id,product_name
0,RTR-200,Wireless Router X200
1,RTR-200P,Wireless Router X200 Pro


Two hits, `X200` and `X200 Pro`, with no ranking between them. A substring check can only tell you "contains" or "does not contain", it has no notion of *closer*, so this naive lookup has no way to prefer one over the other, it would have to pick arbitrarily, by row order or whichever the code happens to check first.

In [10]:
resolved_ambiguous = index.match(queries=pd.DataFrame({"product_name": ["x200"]}), config=config)
resolved_ambiguous[["product_name_candidate", "product_name_score"]]

,product_name_candidate,product_name_score
0,Wireless Router X200,89
1,Wireless Router X200 Pro,88


M|BOX at least makes the ambiguity visible and quantified, scores a point apart, rather than invisible. But notice this is a genuinely different situation from the typo case above: there, one candidate was clearly correct once typos were tolerated. Here, two candidates are legitimately close, and picking one anyway is a real risk, not a resolved one. Turning a score, or a narrow score gap between the top two candidates, into an actual decision, proceed, ask the customer which router they mean, or escalate, is exactly what the next notebook builds.

## 6. The takeaway

Retrieval over a knowledge base of named entities is an entity resolution problem wearing a search problem's clothes. Literal keyword lookup is brittle to typos and blind to ambiguity in exactly the ways this notebook just measured. Resolving the entity with a deterministic, explainable score first, the same principle `01` introduced for tool calling, turns retrieval into a plain, correct lookup by id, and makes the cases where it's *genuinely* ambiguous visible instead of silently guessed.

## Next steps

- **`06-confidence_based_escalation.ipynb`** - turn a score, and a score gap like the one above, into a real proceed/ask/escalate decision
- **`07-validating_llm_extracted_data.ipynb`** - use the same resolution step to catch a model's own extraction mistakes before they become actions